In [2]:
import pandas as pd

df = pd.read_csv("fin_project.csv") 

df["date"] = pd.to_datetime(df["date"], errors="coerce")

mau = df.loc[df["date"].dt.to_period("M") == "2023-11", "user_id"].nunique()

print("MAU (Nov 2023):", mau)

MAU (Nov 2023): 7639


In [3]:
nov = df[df["date"].dt.to_period("M") == "2023-11"].copy()

dau_nov = (
    nov.groupby(nov["date"].dt.date)["user_id"]
       .nunique()
)

print("Средний DAU за ноябрь:", dau_nov.mean())
print("Медианный DAU за ноябрь:", dau_nov.median())
print("Максимальный DAU за ноябрь:", dau_nov.max())
print("Минимальный DAU за ноябрь:", dau_nov.min())

Средний DAU за ноябрь: 560.4666666666667
Медианный DAU за ноябрь: 609.0
Максимальный DAU за ноябрь: 711
Минимальный DAU за ноябрь: 343


In [4]:
df = pd.read_csv("fin_project.csv")   
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date

first_date = df.groupby("user_id")["date"].min()


cohort_users = first_date[first_date == pd.to_datetime("2023-11-01").date()].index


returned_users = df[
    (df["user_id"].isin(cohort_users)) &
    (df["date"] == pd.to_datetime("2023-11-02").date())
]["user_id"].nunique()

cohort_size = len(cohort_users)

retention_d1 = returned_users / cohort_size * 100

print("Cohort size (first day 2023-11-01):", cohort_size)
print("Returned on 2023-11-02:", returned_users)
print("D1 retention (%):", round(retention_d1, 1))


Cohort size (first day 2023-11-01): 623
Returned on 2023-11-02: 166
D1 retention (%): 26.6


In [5]:
df = pd.read_csv("fin_project.csv")  
df["date"] = pd.to_datetime(df["date"])


nov = df[df["date"].dt.month == 11]

total_users = nov["user_id"].nunique()
users_with_views = nov.loc[nov["view_adverts"] > 0, "user_id"].nunique()

conversion = users_with_views / total_users * 100
total_users, users_with_views, conversion

(7639, 3538, 46.31496269145176)

In [6]:
df = pd.read_csv("fin_project.csv")  
df["date"] = pd.to_datetime(df["date"])

nov = df[df["date"].dt.month == 11].copy()

views_per_user = nov.groupby("user_id")["view_adverts"].sum()

avg_views = views_per_user.mean()
avg_views

np.float64(2.8687000916350307)

In [8]:
import pandas as pd
import numpy as np
from scipy import stats

df_ab = pd.read_csv("ab.csv")

df_ab["revenue"] = pd.to_numeric(df_ab["revenue"], errors="coerce").fillna(0)

alpha = 0.05

results = []

for exp in sorted(df_ab["experiment_num"].unique()):
    sub = df_ab[df_ab["experiment_num"] == exp]

    test = sub[sub["experiment_group"] == "test"]["revenue"]
    control = sub[sub["experiment_group"] == "control"]["revenue"]

    arpu_test = test.mean()
    arpu_control = control.mean()
    diff = arpu_test - arpu_control

    t_stat, p_value = stats.ttest_ind(test, control, equal_var=False)

    u_stat, p_u = stats.mannwhitneyu(test, control, alternative="two-sided")

    decision = "Отклоняем H0 (есть разница)" if p_value <= alpha else "Не отклоняем H0 (разницы не нашли)"

    results.append({
        "experiment_num": exp,
        "ARPU_test": arpu_test,
        "ARPU_control": arpu_control,
        "diff_test_minus_control": diff,
        "t_stat": t_stat,
        "p_value_ttest": p_value,
        "p_value_mannwhitney": p_u,
        "decision(alpha=0.05)": decision
    })

res_df = pd.DataFrame(results)
print(res_df)

   experiment_num   ARPU_test  ARPU_control  diff_test_minus_control  \
0               1  665.739583    722.460215               -56.720632   
1               2  332.929167    704.653763              -371.724597   
2               3  998.668750    663.206452               335.462298   

     t_stat  p_value_ttest  p_value_mannwhitney  \
0 -0.400382       0.688966             0.796006   
1 -3.270721       0.001128             0.008453   
2  1.880979       0.060315             0.001001   

                 decision(alpha=0.05)  
0  Не отклоняем H0 (разницы не нашли)  
1         Отклоняем H0 (есть разница)  
2  Не отклоняем H0 (разницы не нашли)  


In [10]:
import pandas as pd

listers = pd.read_csv("list.csv")  

avg_income_per_user = (
    listers.groupby("user_id")["revenue"].sum().mean()
)

avg_income_per_user

np.float64(156.48387096774192)

In [12]:
median_age = (
    listers.groupby("user_id")["age"]
    .first()          
    .median()
)
median_age

28.0